In [2]:
import warnings
import os
warnings.filterwarnings('ignore')
warnings.filterwarnings("ignore", category=UserWarning, module="seqeval")
import seqeval
warnings.filterwarnings("ignore", module=r"seqeval\..*")

In [3]:
import os
import logging
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["DISABLE_TQDM"] = "1"
from transformers.utils import logging
logging.set_verbosity_error()
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()

In [4]:
# Install libraries
%pip install -q transformers datasets seqeval evaluate accelerate --quiet
%pip install -q nltk --quiet

In [5]:
import numpy as np
import torch
import nltk
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    pipeline,
)
import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Device: {device}")
print(f"✅ PyTorch: {torch.__version__}")

✅ Device: cpu
✅ PyTorch: 2.11.0+cu130


In [6]:
# Sample dataset (tokens + POS + chunk tags)

data = [
    {
        "tokens": ["John", "works", "at", "Google"],
        "pos_tags": ["NNP", "VBZ", "IN", "NNP"],
        "chunk_tags": ["B-NP", "B-VP", "B-PP", "B-NP"],
    },
    {
        "tokens": ["She", "loves", "machine", "learning"],
        "pos_tags": ["PRP", "VBZ", "NN", "NN"],
        "chunk_tags": ["B-NP", "B-VP", "I-NP", "I-NP"],
    },
    {
        "tokens": ["I", "am", "learning", "NLP"],
        "pos_tags": ["PRP", "VBP", "VBG", "NNP"],
        "chunk_tags": ["B-NP", "B-VP", "I-VP", "B-NP"],
    },
]

In [7]:
dataset = Dataset.from_list(data)

dataset = DatasetDict({
    "train": dataset,
    "validation": dataset,
    "test": dataset
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],
        num_rows: 3
    })
    validation: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],
        num_rows: 3
    })
    test: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],
        num_rows: 3
    })
})


In [8]:
# Get unique POS and chunk labels

pos_labels = list(set(label for example in data for label in example["pos_tags"]))
chunk_labels = list(set(label for example in data for label in example["chunk_tags"]))

print("POS Labels:", pos_labels)
print("Chunk Labels:", chunk_labels)

POS Labels: ['NNP', 'PRP', 'NN', 'VBZ', 'VBP', 'VBG', 'IN']
Chunk Labels: ['B-VP', 'B-NP', 'B-PP', 'I-NP', 'I-VP']


In [9]:
# Create mappings

pos_label2id = {label: i for i, label in enumerate(pos_labels)}
pos_id2label = {i: label for label, i in pos_label2id.items()}

chunk_label2id = {label: i for i, label in enumerate(chunk_labels)}
chunk_id2label = {i: label for label, i in chunk_label2id.items()}

print("POS mapping:", pos_label2id)
print("Chunk mapping:", chunk_label2id)

POS mapping: {'NNP': 0, 'PRP': 1, 'NN': 2, 'VBZ': 3, 'VBP': 4, 'VBG': 5, 'IN': 6}
Chunk mapping: {'B-VP': 0, 'B-NP': 1, 'B-PP': 2, 'I-NP': 3, 'I-VP': 4}


In [10]:
# Convert labels to numbers

def encode_labels(example):
    example["pos_tags"] = [pos_label2id[label] for label in example["pos_tags"]]
    example["chunk_tags"] = [chunk_label2id[label] for label in example["chunk_tags"]]
    return example

dataset = dataset.map(encode_labels)

dataset["train"][0]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

{'tokens': ['John', 'works', 'at', 'Google'],
 'pos_tags': [0, 3, 6, 0],
 'chunk_tags': [1, 0, 2, 1]}

In [11]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

WARNING: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


In [12]:
def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    word_ids = tokenized_inputs.word_ids()

    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        elif word_idx != previous_word_idx:
            labels.append(example["pos_tags"][word_idx])  # POS tagging
        else:
            labels.append(-100)

        previous_word_idx = word_idx

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [13]:
tokenized_datasets = dataset.map(tokenize_and_align_labels)

tokenized_datasets["train"][0]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

{'tokens': ['John', 'works', 'at', 'Google'],
 'pos_tags': [0, 3, 6, 0],
 'chunk_tags': [1, 0, 2, 1],
 'input_ids': [101, 2198, 2573, 2012, 8224, 102],
 'token_type_ids': [0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1],
 'labels': [-100, 0, 3, 6, 0, -100]}

In [14]:
# Number of POS labels

num_labels = len(pos_labels)

print("Number of labels:", num_labels)

Number of labels: 7


In [15]:
# Load BERT model for token classification

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    id2label=pos_id2label,
    label2id=pos_label2id
)

model.to(device)

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

In [16]:
# Handles dynamic padding

data_collator = DataCollatorForTokenClassification(tokenizer)

In [17]:
# Training configuration

training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=3,   # small dataset → slightly more epochs
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    learning_rate=2e-5,

    logging_steps=10,

    do_train=True,
    do_eval=True
)

In [18]:
# Load seqeval metric

metric = evaluate.load("seqeval")

In [19]:
# Metrics function

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for pred, lab in zip(predictions, labels):
        curr_preds = []
        curr_labels = []

        for p_val, l_val in zip(pred, lab):
            if l_val != -100:
                curr_preds.append(pos_id2label[p_val])
                curr_labels.append(pos_id2label[l_val])

        true_predictions.append(curr_preds)
        true_labels.append(curr_labels)

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [20]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [21]:
trainer.train()

{'train_runtime': '22.84', 'train_samples_per_second': '0.394', 'train_steps_per_second': '0.263', 'train_loss': '1.809', 'epoch': '3'}


TrainOutput(global_step=6, training_loss=1.808776060740153, metrics={'train_runtime': 22.8359, 'train_samples_per_second': 0.394, 'train_steps_per_second': 0.263, 'train_loss': 1.808776060740153, 'epoch': 3.0})

In [22]:
# Evaluate model

results = trainer.evaluate()

print("Evaluation Results:")
for key, value in results.items():
    print(f"{key}: {value}")

{'eval_loss': '1.539', 'eval_precision': '0.7', 'eval_recall': '0.6364', 'eval_f1': '0.6667', 'eval_accuracy': '0.75', 'eval_runtime': '0.2022', 'eval_samples_per_second': '14.83', 'eval_steps_per_second': '9.889', 'epoch': '3'}
Evaluation Results:
eval_loss: 1.5385231971740723
eval_precision: 0.7
eval_recall: 0.6363636363636364
eval_f1: 0.6666666666666666
eval_accuracy: 0.75
eval_runtime: 0.2022
eval_samples_per_second: 14.834
eval_steps_per_second: 9.889
epoch: 3.0


In [23]:
# Create inference pipeline

nlp = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if device == "cuda" else -1
)

In [24]:
# Test sentence

sentence = "John works at Google in California"

output = nlp(sentence)

for token in output:
    print(token)

{'entity': 'PRP', 'score': np.float32(0.17087016), 'index': 1, 'word': 'john', 'start': 0, 'end': 4}
{'entity': 'VBZ', 'score': np.float32(0.23121801), 'index': 2, 'word': 'works', 'start': 5, 'end': 10}
{'entity': 'NNP', 'score': np.float32(0.20417531), 'index': 3, 'word': 'at', 'start': 11, 'end': 13}
{'entity': 'NNP', 'score': np.float32(0.3202209), 'index': 4, 'word': 'google', 'start': 14, 'end': 20}
{'entity': 'NN', 'score': np.float32(0.19328403), 'index': 5, 'word': 'in', 'start': 21, 'end': 23}
{'entity': 'NNP', 'score': np.float32(0.22156453), 'index': 6, 'word': 'california', 'start': 24, 'end': 34}


In [25]:
# Test sentence

sentence = "John works at Google in California"

output = nlp(sentence)

for token in output:
    print(token)

{'entity': 'PRP', 'score': np.float32(0.17087016), 'index': 1, 'word': 'john', 'start': 0, 'end': 4}
{'entity': 'VBZ', 'score': np.float32(0.23121801), 'index': 2, 'word': 'works', 'start': 5, 'end': 10}
{'entity': 'NNP', 'score': np.float32(0.20417531), 'index': 3, 'word': 'at', 'start': 11, 'end': 13}
{'entity': 'NNP', 'score': np.float32(0.3202209), 'index': 4, 'word': 'google', 'start': 14, 'end': 20}
{'entity': 'NN', 'score': np.float32(0.19328403), 'index': 5, 'word': 'in', 'start': 21, 'end': 23}
{'entity': 'NNP', 'score': np.float32(0.22156453), 'index': 6, 'word': 'california', 'start': 24, 'end': 34}


In [26]:
# Clean readable output

for token in output:
    print(f"{token['word']} → {token['entity']}")

john → PRP
works → VBZ
at → NNP
google → NNP
in → NN
california → NNP
